In [ ]:
# ====================================
# Step 1: Google Driveをマウント
# ====================================
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ====================================
# Step 2: パス設定
# ====================================
import os

# Drive上のプロジェクトディレクトリを指定
BASE_DIR = "/content/drive/Shareddrives/nsketch/transcribe"
SOURCE_DIR = os.path.join(BASE_DIR, "input")        # 音声ファイルの入力フォルダ
OUTPUT_DIR = os.path.join(BASE_DIR, "output")       # テキスト出力先
DONE_DIR = os.path.join(BASE_DIR, "done")           # 処理済みフォルダ
TEMP_DIR = os.path.join(BASE_DIR, "tmp_audio")      # 一時的にWAVへ変換した音声の保存先

# フォルダがなければ作成
for directory in [SOURCE_DIR, OUTPUT_DIR, DONE_DIR, TEMP_DIR]:
    os.makedirs(directory, exist_ok=True)

In [ ]:
# ====================================
# Step 3: 必要ライブラリをインストール
# ====================================
import importlib.metadata as metadata
import os
import subprocess
import sys

BASE_DIR = "/content/drive/Shareddrives/nsketch/transcribe"
RUNTIME_RESTART_MARKER = "/tmp/colab_restart_required"

def pip_install(*packages):
    command = [sys.executable, "-m", "pip", "install", "-U", "--no-cache-dir", *packages]
    print("Running:", " ".join(command))
    subprocess.check_call(command)

def get_installed_version(package_name):
    try:
        return metadata.version(package_name)
    except metadata.PackageNotFoundError:
        return None

!apt-get -y install ffmpeg

# Colab に入っている torch に合わせて torchaudio の版を揃える
try:
    torch_version = metadata.version("torch").split("+")[0]
except metadata.PackageNotFoundError:
    torch_version = "2.6.0"

print("torch version:", torch_version)

required_versions = {
    "numpy": "1.26.4",
    "numba": "0.60.0",
    "more-itertools": None,
    "tiktoken": None,
    "tqdm": None,
    "pyannote.audio": "3.1.1",
    "python-dotenv": "1.0.1",
    "onnxruntime": "1.20.1",
    "torchaudio": torch_version,
}

packages_to_install = []
for package_name, expected_version in required_versions.items():
    installed_version = get_installed_version(package_name)
    if expected_version is None:
        if installed_version is None:
            packages_to_install.append(package_name)
    elif installed_version != expected_version:
        packages_to_install.append(f"{package_name}=={expected_version}")

restart_required = False

if packages_to_install:
    print("Installing/updating packages:", packages_to_install)
    pip_install("--force-reinstall", *packages_to_install)
    restart_required = True
else:
    print("Required packages are already installed.")

local_whisper_dir = os.path.join(BASE_DIR, "whisper")
if os.path.isdir(local_whisper_dir):
    print(f"Installing whisper from local repo: {local_whisper_dir}")
    pip_install("--no-deps", local_whisper_dir)
else:
    print("Local whisper repo not found. Falling back to PyPI / GitHub install.")
    try:
        pip_install("openai-whisper")
    except subprocess.CalledProcessError:
        pip_install("git+https://github.com/openai/whisper.git")

if restart_required:
    with open(RUNTIME_RESTART_MARKER, "w", encoding="utf-8") as marker_file:
        marker_file.write("Step 3 updated binary dependencies. Runtime restart is required.\n")

    raise SystemExit(
        "依存ライブラリを更新しました。Colab のランタイムを再起動し、再起動後は Step 1 を実行してから Step 4 に進んでください。"
    )

print("Environment is ready.")

## Hugging Faceトークンの準備

`pyannote.audio` の話者分離モデルを初回ロードするには Hugging Face のアクセストークンが必要です。

優先順は次の通りです。

1. Colab の Secrets に設定した `HF_TOKEN`
2. Drive 上の `.env` に設定した `HF_TOKEN`
3. 実行環境の環境変数 `HF_TOKEN`

`.env` を使う場合は、たとえば `/content/drive/Shareddrives/nsketch/transcribe/.env` に次のように保存します。

```env
HF_TOKEN=hf_xxxxxxxxxxxxxxxxx
```

Step 3 では `numpy==1.26.4` と `numba==0.60.0` を含む依存関係を確認し、不足や版ずれがある場合だけ再インストールします。
依存関係を更新した直後は Colab のランタイムを必ず再起動してください。
再起動後は Drive を再マウントするために Step 1 を実行し、その後は Step 4 から再開できます。
Step 2 と Step 3 は、パス設定や依存確認をやり直したい場合にだけ再実行してください。
再起動せずに Step 4 へ進むと、NumPy や pyannote の import が壊れた状態のまま残ることがあります。

`whisper` は Drive 上の `whisper/` リポジトリを優先してインストールします。
そのため `openai-whisper==20231117` のビルド失敗を回避できます。

In [ ]:
# ====================================
# Step 4: Whisperと話者分離モデルをロード
# ====================================
import os

BASE_DIR = "/content/drive/Shareddrives/nsketch/transcribe"
CACHE_DIR = os.path.join(BASE_DIR, "model_cache")
ENV_FILE = os.path.join(BASE_DIR, ".env")
RUNTIME_RESTART_MARKER = "/tmp/colab_restart_required"

if not os.path.exists(BASE_DIR):
    raise RuntimeError("Drive が未マウントです。Step 1 を実行してください。")

if os.path.exists(RUNTIME_RESTART_MARKER):
    raise RuntimeError(
        "Step 3 で依存ライブラリが更新されています。Colab のランタイムを再起動し、Step 1 を実行してから Step 4 に進んでください。"
    )

try:
    import numpy as np
    import torch
    import whisper
except ValueError as error:
    if "numpy.dtype size changed" in str(error):
        raise RuntimeError(
            "NumPy のバイナリ不整合が残っています。Colab のランタイムを再起動し、Step 1 を実行してから Step 4 に進んでください。"
        ) from error
    raise

from dotenv import load_dotenv

try:
    from pyannote.audio import Pipeline, Audio
except Exception as error:
    raise RuntimeError(
        "pyannote.audio の import に失敗しました。Step 3 のあとに Colab ランタイムを再起動し、Step 1 を実行してから Step 4 に進んでください。"
    ) from error

os.environ["XDG_CACHE_HOME"] = CACHE_DIR

print("NumPy version:", np.__version__)

if os.path.exists(ENV_FILE):
    load_dotenv(ENV_FILE)
    print(f"Loaded .env: {ENV_FILE}")
else:
    print(f".env not found: {ENV_FILE}")

try:
    from google.colab import userdata
except ImportError:
    userdata = None

hf_token = None

if userdata is not None:
    try:
        hf_token = userdata.get("HF_TOKEN")
        if hf_token:
            print("Using HF_TOKEN from Colab secrets")
    except Exception:
        hf_token = None

if not hf_token:
    hf_token = os.environ.get("HF_TOKEN")
    if hf_token:
        print("Using HF_TOKEN from environment or .env")

if not hf_token:
    raise ValueError(
        "HF_TOKEN が見つかりません。Colab の Secrets または Drive 上の .env に設定してください。"
    )

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

whisper_model = whisper.load_model("medium")
diarization_pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1",
    use_auth_token=hf_token,
    cache_dir=CACHE_DIR,
)
diarization_pipeline.to(torch.device(device))

audio_cropper = Audio(sample_rate=16000, mono=True)
NUM_SPEAKERS = 2

In [ ]:
# ====================================
# Step 5: 話者分離 + 文字起こし関数
# ====================================
import os
import shutil
import subprocess

BASE_DIR = "/content/drive/Shareddrives/nsketch/transcribe"
OUTPUT_DIR = os.path.join(BASE_DIR, "output")
DONE_DIR = os.path.join(BASE_DIR, "done")
TEMP_DIR = os.path.join(BASE_DIR, "tmp_audio")

for directory in [OUTPUT_DIR, DONE_DIR, TEMP_DIR]:
    os.makedirs(directory, exist_ok=True)

def format_timestamp(seconds):
    total_milliseconds = int(round(float(seconds) * 1000))
    hours, remainder = divmod(total_milliseconds, 3600000)
    minutes, remainder = divmod(remainder, 60000)
    secs, milliseconds = divmod(remainder, 1000)
    if hours:
        return f"{hours:02d}:{minutes:02d}:{secs:02d}.{milliseconds:03d}"
    return f"{minutes:02d}:{secs:02d}.{milliseconds:03d}"

def ensure_wav(file_path):
    if file_path.lower().endswith(".wav"):
        return file_path, False

    base_name = os.path.splitext(os.path.basename(file_path))[0]
    wav_path = os.path.join(TEMP_DIR, base_name + ".wav")

    command = [
        "ffmpeg",
        "-y",
        "-i", file_path,
        "-ar", "16000",
        "-ac", "1",
        wav_path,
    ]
    subprocess.run(command, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    return wav_path, True

def diarize_and_transcribe(file_path, num_speakers=None):
    print(f"Processing: {file_path}")
    wav_path, temporary_file_created = ensure_wav(file_path)

    try:
        diarization = diarization_pipeline(wav_path, num_speakers=num_speakers)
        results = []

        for segment, _, speaker in diarization.itertracks(yield_label=True):
            if (segment.end - segment.start) < 0.3:
                continue

            waveform, _ = audio_cropper.crop(wav_path, segment)
            transcription = whisper_model.transcribe(
                waveform.squeeze().numpy(),
                language="ja",
                fp16=torch.cuda.is_available(),
                condition_on_previous_text=False,
            )
            text = (transcription.get("text") or "").strip()
            if not text:
                continue

            row = {
                "speaker": speaker,
                "start": float(segment.start),
                "end": float(segment.end),
                "text": text,
            }
            results.append(row)
            print(
                f"[{row['start']:.1f}s - {row['end']:.1f}s] {row['speaker']}: {row['text']}"
            )

        return results
    finally:
        if temporary_file_created and os.path.exists(wav_path):
            os.remove(wav_path)

def move_to_done(file_path):
    destination_path = os.path.join(DONE_DIR, os.path.basename(file_path))
    shutil.move(file_path, destination_path)

def save_diarized_transcript(rows, src_filename):
    if not rows:
        print(f"No transcript rows produced: {src_filename}")
        return

    name, _ = os.path.splitext(os.path.basename(src_filename))
    output_path = os.path.join(OUTPUT_DIR, name + ".txt")

    with open(output_path, "w", encoding="utf-8") as file_handle:
        for row in rows:
            start_label = format_timestamp(row["start"])
            end_label = format_timestamp(row["end"])
            file_handle.write(
                f"[{start_label} - {end_label}] {row['speaker']}: {row['text']}\n"
            )

    print(f"Saved: {output_path}")

In [ ]:
# ====================================
# Step 6: 自動監視ループ
# ====================================
import os
import time

BASE_DIR = "/content/drive/Shareddrives/nsketch/transcribe"
SOURCE_DIR = os.path.join(BASE_DIR, "input")
os.makedirs(SOURCE_DIR, exist_ok=True)

SUPPORTED_EXTENSIONS = (".mp3", ".wav", ".m4a")

print("Watching folder:", SOURCE_DIR)
print("Expected speakers:", NUM_SPEAKERS)

while True:
    files = [
        filename
        for filename in os.listdir(SOURCE_DIR)
        if filename.lower().endswith(SUPPORTED_EXTENSIONS)
    ]

    if not files:
        print("No new files. Waiting 30 seconds...")
        time.sleep(30)
        continue

    for filename in files:
        file_path = os.path.join(SOURCE_DIR, filename)
        try:
            rows = diarize_and_transcribe(file_path, num_speakers=NUM_SPEAKERS)
            save_diarized_transcript(rows, filename)
            move_to_done(file_path)
        except Exception as error:
            print(f"Error processing {filename}: {error}")

    print("Cycle complete. Waiting 30 seconds...")
    time.sleep(30)